# SO SÁNH VÀ LỰA CHỌN MÔ HÌNH TỐT HƠN CHO DỰ ĐOÁN
So sánh điểm số Cross-Validation (CV) tốt nhất của hai mô hình để chọn ra mô hình có hiệu suất nội bộ cao nhất.

    -Random Forest (RF): Điểm CV tốt nhất là RF_Best_Score
    -Gradient Boosting (GB): Điểm CV tốt nhất là GB_Best_Score
    
Chiến lược: Mô hình có điểm CV cao hơn (hoặc mô hình cho kết quả tốt hơn khi nộp thử lên Kaggle) sẽ là mô hình chính.

In [2]:
import pandas as pd
import numpy as np
import joblib # Cần import joblib để load model

## 1. Nạp dữ liệu và Model

In [3]:
try:
    X_test = pd.read_csv('X_test_processed.csv')
    test_ids = pd.read_csv('test_ids.csv')['PassengerId']
    
    # NẠP CÁC MÔ HÌNH ĐÃ LƯU
    rf_final_model = joblib.load('rf_final_model.pkl')
    gb_final_model = joblib.load('gb_final_model.pkl')
    
    print("Đã nạp dữ liệu và mô hình thành công.")
except FileNotFoundError:
    print("LỖI: Không tìm thấy file dữ liệu/mô hình. Hãy chạy lại các file preprocessing.ipynb, model1.ipynb, và model2.ipynb.")
    import sys; sys.exit()

Đã nạp dữ liệu và mô hình thành công.


## 2. Kết hợp Mô hình (Ensembling)
**Kỹ thuật đề xuất: Soft Voting Ensemble**

*   Huấn luyện lại cả hai mô hình (RF và GB) trên toàn bộ `X_train` với các tham số tối ưu.
*   Sử dụng cả hai mô hình để dự đoán **xác suất** (`predict_proba`) trên `X_test`.
*   Tính trung bình xác suất của hai mô hình và quyết định lớp cuối cùng (Survived/Not Survived) dựa trên xác suất trung bình này.

In [4]:
# Lấy xác suất sống sót (cột thứ 2)
# Cột 0 là xác suất Not Survived, cột 1 là xác suất Survived
rf_proba = rf_final_model.predict_proba(X_test)[:, 1]
gb_proba = gb_final_model.predict_proba(X_test)[:, 1]

# Tính trung bình xác suất (Soft Voting)
avg_proba = (rf_proba + gb_proba) / 2

# Chuyển xác suất trung bình thành dự đoán nhị phân (threshold = 0.5)
ensemble_predictions = (avg_proba >= 0.5).astype(int)

## 3. Lưu mô hình

In [5]:
submission_ensemble_df = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': ensemble_predictions
})

submission_ensemble_df.to_csv('submission_ensemble_final.csv', index=False)
print("\nĐÃ LƯU: submission_ensemble_final.csv (Mô hình Ensemble)")


ĐÃ LƯU: submission_ensemble_final.csv (Mô hình Ensemble)


# Kết Thúc